# Amharic ASR Benchmark Notebook

This Colab notebook benchmarks the Hugging Face ASR model `b1n1yam/shook-medium-amharic-2k`.

Default evaluation setup:
- Model: `b1n1yam/shook-medium-amharic-2k`
- Dataset: `mteb/fleurs`
- Language config: `am_et`
- Split: `test`
- Sample size: `50` utterances by default

Outputs:
- `manifest_used.csv`
- `utterance_level_results.csv`
- `summary_metrics.csv`
- `benchmark_summary.json`

Notes:
- This notebook is intentionally benchmark-focused only.
- You can switch to your own dataset by providing a manifest CSV with `audio_path` and `reference` columns.

In [ ]:
%pip -q install "datasets[audio]<4" transformers accelerate jiwer soundfile librosa scipy seaborn pandas numpy torch tqdm


In [ ]:
import json
import os
import random
import re
import time
import unicodedata
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import soundfile as sf
import torch
from datasets import load_dataset
from IPython.display import Markdown, display
from jiwer import process_characters, process_words
from tqdm.auto import tqdm
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.max_colwidth", 220)

MODEL_ID = os.getenv("MODEL_ID", "b1n1yam/shook-medium-amharic-2k")
DATASET_ID = os.getenv("DATASET_ID", "mteb/fleurs")
DATASET_CONFIG = os.getenv("DATASET_CONFIG", "am_et")
DATASET_SPLIT = os.getenv("DATASET_SPLIT", "test")
SAMPLE_SIZE = int(os.getenv("SAMPLE_SIZE", "50"))
SEED = int(os.getenv("SEED", "42"))
CUSTOM_MANIFEST_CSV = os.getenv("CUSTOM_MANIFEST_CSV", "").strip()
MAX_NEW_TOKENS = int(os.getenv("MAX_NEW_TOKENS", "225"))

OUTPUT_DIR = Path("amharic_shook_benchmark")
AUDIO_DIR = OUTPUT_DIR / "audio_clean"
NOISY_AUDIO_DIR = OUTPUT_DIR / "audio_noisy"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
NOISY_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

print(f"Model: {MODEL_ID}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
if CUSTOM_MANIFEST_CSV:
    print(f"Custom manifest mode: {CUSTOM_MANIFEST_CSV}")
else:
    print(f"Dataset mode: {DATASET_ID} / {DATASET_CONFIG} / {DATASET_SPLIT}")


In [ ]:
def normalize_text(text: str, remove_punct: bool = False) -> str:
    text = unicodedata.normalize("NFC", text or "")
    text = re.sub(r"\s+", " ", text.strip().lower())
    if remove_punct:
        text = "".join(ch for ch in text if not unicodedata.category(ch).startswith("P"))
        text = re.sub(r"\s+", " ", text).strip()
    return text


def bootstrap_ci(values, n_boot: int = 2000, ci: float = 95.0, seed: int = SEED):
    arr = np.asarray([value for value in values if pd.notna(value)], dtype=float)
    if arr.size == 0:
        return (np.nan, np.nan)
    if arr.size == 1:
        return (float(arr[0]), float(arr[0]))
    rng = np.random.default_rng(seed)
    means = np.empty(n_boot, dtype=float)
    for idx in range(n_boot):
        sample = rng.choice(arr, size=arr.size, replace=True)
        means[idx] = float(np.mean(sample))
    alpha = (100 - ci) / 2
    return float(np.percentile(means, alpha)), float(np.percentile(means, 100 - alpha))


def summarize_metric(values, prefix: str):
    arr = np.asarray([value for value in values if pd.notna(value)], dtype=float)
    if arr.size == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_p95": np.nan,
            f"{prefix}_ci_low": np.nan,
            f"{prefix}_ci_high": np.nan,
        }
    ci_low, ci_high = bootstrap_ci(arr)
    return {
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_median": float(np.median(arr)),
        f"{prefix}_p95": float(np.percentile(arr, 95)),
        f"{prefix}_ci_low": ci_low,
        f"{prefix}_ci_high": ci_high,
    }


def compute_error_metrics(reference: str, prediction: str):
    reference = reference or ""
    prediction = prediction or ""
    raw_word = process_words(reference, prediction)
    raw_char = process_characters(reference, prediction)
    normalized_reference = normalize_text(reference, remove_punct=True)
    normalized_prediction = normalize_text(prediction, remove_punct=True)
    normalized_word = process_words(normalized_reference, normalized_prediction)
    normalized_char = process_characters(normalized_reference, normalized_prediction)
    return {
        "reference_raw": reference,
        "prediction_raw": prediction,
        "reference_normalized": normalized_reference,
        "prediction_normalized": normalized_prediction,
        "wer_raw": float(raw_word.wer),
        "cer_raw": float(raw_char.cer),
        "wer_normalized": float(normalized_word.wer),
        "cer_normalized": float(normalized_char.cer),
    }


def load_and_resample_audio(audio_path: Path, target_sr: int = 16000):
    audio, sampling_rate = sf.read(audio_path, dtype="float32")
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)
    if sampling_rate != target_sr:
        audio = librosa.resample(audio, orig_sr=sampling_rate, target_sr=target_sr)
        sampling_rate = target_sr
    peak = np.max(np.abs(audio)) if audio.size else 0.0
    if peak > 1.0:
        audio = audio / peak
    return audio.astype(np.float32), int(sampling_rate)


def prepare_manifest_from_fleurs(sample_size: int = SAMPLE_SIZE, seed: int = SEED) -> pd.DataFrame:
    dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=DATASET_SPLIT)
    if sample_size > len(dataset):
        raise ValueError(f"Requested {sample_size} samples but split only has {len(dataset)} rows.")
    rng = random.Random(seed)
    selected_indices = sorted(rng.sample(range(len(dataset)), sample_size))
    rows = []
    for dataset_row_index in tqdm(selected_indices, desc="Preparing FLEURS Amharic"):
        row = dataset[int(dataset_row_index)]
        utterance_id = f"am-{row['id']}"
        audio = row["audio"]
        audio_array = np.asarray(audio["array"], dtype=np.float32)
        if audio_array.ndim > 1:
            audio_array = np.mean(audio_array, axis=1)
        sampling_rate = int(audio["sampling_rate"])
        if sampling_rate != 16000:
            audio_array = librosa.resample(audio_array, orig_sr=sampling_rate, target_sr=16000)
            sampling_rate = 16000
        audio_path = AUDIO_DIR / f"{utterance_id}.wav"
        sf.write(audio_path, audio_array, sampling_rate)
        rows.append(
            {
                "utterance_id": utterance_id,
                "source": "mteb/fleurs",
                "dataset_config": DATASET_CONFIG,
                "dataset_split": DATASET_SPLIT,
                "dataset_row_index": int(dataset_row_index),
                "reference": row["transcription"],
                "raw_reference": row.get("raw_transcription", row["transcription"]),
                "duration_seconds": round(len(audio_array) / sampling_rate, 6),
                "audio_path": str(audio_path),
            }
        )
    return pd.DataFrame(rows).sort_values("dataset_row_index").reset_index(drop=True)


def prepare_manifest_from_custom_csv(csv_path: str) -> pd.DataFrame:
    manifest = pd.read_csv(csv_path)
    required_columns = {"audio_path", "reference"}
    missing = required_columns - set(manifest.columns)
    if missing:
        raise ValueError(f"Custom manifest is missing required columns: {sorted(missing)}")
    if "utterance_id" not in manifest.columns:
        manifest["utterance_id"] = [f"custom-{idx:05d}" for idx in range(len(manifest))]
    manifest["audio_path"] = manifest["audio_path"].astype(str)
    durations = []
    normalized_paths = []
    for audio_path_str in tqdm(manifest["audio_path"].tolist(), desc="Validating custom manifest"):
        audio_path = Path(audio_path_str).expanduser()
        if not audio_path.exists():
            raise FileNotFoundError(f"Audio file not found: {audio_path}")
        audio_array, sr = load_and_resample_audio(audio_path)
        normalized_audio_path = AUDIO_DIR / f"{audio_path.stem}.wav"
        sf.write(normalized_audio_path, audio_array, sr)
        normalized_paths.append(str(normalized_audio_path))
        durations.append(round(len(audio_array) / sr, 6))
    manifest = manifest.copy()
    manifest["audio_path"] = normalized_paths
    manifest["duration_seconds"] = durations
    manifest["source"] = manifest.get("source", "custom_manifest")
    manifest["dataset_config"] = manifest.get("dataset_config", "custom")
    manifest["dataset_split"] = manifest.get("dataset_split", "custom")
    return manifest.reset_index(drop=True)


def load_asr_runtime():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    torch_dtype = torch.float16 if device.type == "cuda" else torch.float32

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID,
        torch_dtype=torch_dtype,
        low_cpu_mem_usage=True,
        use_safetensors=True,
    )
    model = model.to(device)
    model.eval()

    if hasattr(processor, "get_decoder_prompt_ids"):
        model.generation_config.forced_decoder_ids = processor.get_decoder_prompt_ids(
            language="amharic",
            task="transcribe",
        )

    return {
        "processor": processor,
        "model": model,
        "device": device,
        "torch_dtype": torch_dtype,
    }


def transcribe_with_model(asr_runtime, audio_path: Path):
    started = time.perf_counter()
    audio_array, sampling_rate = load_and_resample_audio(audio_path)
    processor = asr_runtime["processor"]
    model = asr_runtime["model"]
    device = asr_runtime["device"]
    torch_dtype = asr_runtime["torch_dtype"]

    inputs = processor(audio_array, sampling_rate=sampling_rate, return_tensors="pt")
    input_features = inputs["input_features"].to(device=device, dtype=torch_dtype)

    generation_kwargs = {
        "max_new_tokens": MAX_NEW_TOKENS,
    }
    if "attention_mask" in inputs:
        generation_kwargs["attention_mask"] = inputs["attention_mask"].to(device=device)

    with torch.inference_mode():
        generated_ids = model.generate(
            input_features=input_features,
            **generation_kwargs,
        )

    text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    elapsed_s = time.perf_counter() - started
    return {
        "transcription": text.strip(),
        "inference_seconds": float(elapsed_s),
    }


def run_benchmark(manifest: pd.DataFrame, asr_runtime) -> pd.DataFrame:
    rows = []
    for _, row in tqdm(manifest.iterrows(), total=len(manifest), desc="Benchmarking model"):
        prediction = transcribe_with_model(asr_runtime, Path(row["audio_path"]))
        metrics = compute_error_metrics(row["reference"], prediction["transcription"])
        rtf = prediction["inference_seconds"] / row["duration_seconds"] if row["duration_seconds"] else np.nan
        rows.append(
            {
                **row.to_dict(),
                **prediction,
                **metrics,
                "rtf": float(rtf) if pd.notna(rtf) else np.nan,
            }
        )
    return pd.DataFrame(rows)


def build_summary_table(results_df: pd.DataFrame) -> pd.DataFrame:
    row = {
        "model_id": MODEL_ID,
        "requests": int(len(results_df)),
    }
    for metric_name in ["wer_normalized", "cer_normalized", "wer_raw", "cer_raw", "inference_seconds", "rtf"]:
        row.update(summarize_metric(results_df[metric_name], metric_name))
    return pd.DataFrame([row])


def render_outputs(results_df: pd.DataFrame, summary_df: pd.DataFrame):
    display(Markdown("## Summary metrics"))
    display(summary_df)

    plt.figure(figsize=(10, 5))
    metric_df = pd.DataFrame(
        {
            "Metric": ["Normalized WER", "Normalized CER"],
            "Value": [
                summary_df.iloc[0]["wer_normalized_mean"],
                summary_df.iloc[0]["cer_normalized_mean"],
            ],
        }
    )
    sns.barplot(data=metric_df, x="Metric", y="Value", palette=["#0f766e", "#1d4ed8"])
    plt.title("Amharic benchmark headline metrics")
    plt.ylabel("Score")
    plt.xlabel("")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    sns.violinplot(data=results_df, y="rtf", inner="quartile", color="#ea580c")
    plt.title("Real-time factor distribution")
    plt.ylabel("RTF")
    plt.xlabel("")
    plt.tight_layout()
    plt.show()

    failures = results_df.sort_values(["wer_normalized", "cer_normalized"], ascending=[False, False]).head(10)
    display(Markdown("## Top failure examples"))
    display(
        failures[
            [
                "utterance_id",
                "reference_raw",
                "prediction_raw",
                "wer_normalized",
                "cer_normalized",
                "duration_seconds",
                "inference_seconds",
            ]
        ]
    )


In [ ]:
if CUSTOM_MANIFEST_CSV:
    clean_manifest = prepare_manifest_from_custom_csv(CUSTOM_MANIFEST_CSV)
else:
    clean_manifest = prepare_manifest_from_fleurs(sample_size=SAMPLE_SIZE, seed=SEED)

clean_manifest.to_csv(OUTPUT_DIR / "manifest_used.csv", index=False)
display(Markdown("## Manifest preview"))
display(clean_manifest.head())
print(f"Saved manifest to {(OUTPUT_DIR / 'manifest_used.csv').resolve()}")


In [ ]:
asr_runtime = load_asr_runtime()
print("Model runtime loaded successfully.")


In [ ]:
benchmark_results = run_benchmark(clean_manifest, asr_runtime)
summary_metrics = build_summary_table(benchmark_results)

benchmark_results.to_csv(OUTPUT_DIR / "utterance_level_results.csv", index=False)
summary_metrics.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)

benchmark_summary = {
    "model_id": MODEL_ID,
    "dataset_id": DATASET_ID if not CUSTOM_MANIFEST_CSV else "custom_manifest",
    "dataset_config": DATASET_CONFIG if not CUSTOM_MANIFEST_CSV else "custom",
    "dataset_split": DATASET_SPLIT if not CUSTOM_MANIFEST_CSV else "custom",
    "sample_size": int(len(clean_manifest)),
    "headline_metrics": summary_metrics.to_dict(orient="records"),
}
with (OUTPUT_DIR / "benchmark_summary.json").open("w", encoding="utf-8") as fp:
    json.dump(benchmark_summary, fp, ensure_ascii=False, indent=2)

render_outputs(benchmark_results, summary_metrics)

print(f"Saved utterance-level results to {(OUTPUT_DIR / 'utterance_level_results.csv').resolve()}")
print(f"Saved summary metrics to {(OUTPUT_DIR / 'summary_metrics.csv').resolve()}")
print(f"Saved benchmark summary to {(OUTPUT_DIR / 'benchmark_summary.json').resolve()}")
